# Titanic EDA

Profile the Titanic dataset, clean it using the assignment's missing-value thresholds, and build the visual story. The raw frame is loaded once and written to `titanic.csv`. Modeling must read that file and must not call `sns.load_dataset` again.



In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
CSV_PATH = ROOT / "titanic.csv"
FIG = ROOT / "figures"
SUMMARY_PATH = ROOT / "eda_summary.json"
FIG.mkdir(exist_ok=True)

STORY_COLUMNS = ["survived", "pclass", "age", "sibsp", "parch", "fare"]



## Load once and save the offline fallback

If `titanic.csv` already exists, reuse it. Otherwise load from Seaborn once and commit that file for offline grading.



In [ ]:
if CSV_PATH.exists():
    raw = pd.read_csv(CSV_PATH)
    print(f"Reusing committed offline file {CSV_PATH.name}")
else:
    raw = sns.load_dataset("titanic")
    raw.to_csv(CSV_PATH, index=False)
    print(f"Loaded Titanic once via sns.load_dataset and saved {CSV_PATH.name}")

print("shape", raw.shape)
raw.info()
raw.describe(include="all")



## Missing values

Rule used below:

- under 5% missing → drop those rows
- 5%–30% missing → impute
- so incomplete that imputation would invent most of the column → drop the column (or encode missing as its own category) and justify in writing



In [ ]:
missing_pct = (raw.isna().mean() * 100).round(4)
missing = {col: float(missing_pct[col]) for col in missing_pct.index if missing_pct[col] > 0}
print("missing_pct", missing)

decisions = []
cleaned = raw.copy()
drop_rows_for, impute_cols, drop_cols = [], [], []

for column, pct in missing.items():
    if pct < 5:
        action = "drop_rows"
        drop_rows_for.append(column)
    elif pct <= 30:
        action = "median_impute"
        impute_cols.append(column)
    else:
        action = "drop_column"
        drop_cols.append(column)
    decisions.append({"column": column, "missing_pct": pct, "action": action})

if drop_cols:
    cleaned = cleaned.drop(columns=drop_cols)
if drop_rows_for:
    cleaned = cleaned.dropna(subset=drop_rows_for)

for column in impute_cols:
    if column not in cleaned.columns:
        continue
    if pd.api.types.is_numeric_dtype(cleaned[column]):
        median = float(cleaned[column].median())
        cleaned[column] = cleaned[column].fillna(median)
        for item in decisions:
            if item["column"] == column:
                item["impute_value"] = median
    else:
        mode = cleaned[column].mode(dropna=True).iloc[0]
        cleaned[column] = cleaned[column].fillna(mode)
        for item in decisions:
            if item["column"] == column:
                item["impute_value"] = str(mode)

cleaned = cleaned.reset_index(drop=True)
pd.DataFrame(decisions)



**Decisions.** `age` is about 19.9% missing → median-impute (median 28.0). `embarked` and `embark_town` are about 0.22% missing → drop those two rows. `deck` is about 77.2% missing → drop the column, because filling a cabin letter for most passengers would invent values rather than recover them. That leaves 889 rows.



## Univariate analysis: age and fare

Histograms, box plots, IQR outlier counts, and the mean / median / mode ordering for fare.



In [ ]:
def iqr_outlier_count(series: pd.Series) -> dict:
    values = series.dropna()
    q1 = float(values.quantile(0.25))
    q3 = float(values.quantile(0.75))
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((values < low) | (values > high)).sum())
    return {"q1": q1, "q3": q3, "iqr": iqr, "low": low, "high": high, "outliers": count}

age_outliers = iqr_outlier_count(raw["age"])
fare_outliers = iqr_outlier_count(raw["fare"])
print("age_outliers", age_outliers)
print("fare_outliers", fare_outliers)

fare = raw["fare"].dropna()
fare_stats = {
    "mean": float(fare.mean()),
    "median": float(fare.median()),
    "mode": float(fare.mode().iloc[0]),
}
fare_stats["shape"] = (
    "right-skewed" if fare_stats["mean"] > fare_stats["median"]
    else "left-skewed" if fare_stats["mean"] < fare_stats["median"]
    else "symmetric"
)
print("fare_shape", fare_stats)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
sns.histplot(cleaned["age"], bins=20, ax=axes[0, 0])
axes[0, 0].set_title("Age histogram")
sns.boxplot(x=cleaned["age"], ax=axes[0, 1])
axes[0, 1].set_title("Age box plot")
sns.histplot(cleaned["fare"], bins=20, ax=axes[1, 0])
axes[1, 0].set_title("Fare histogram")
sns.boxplot(x=cleaned["fare"], ax=axes[1, 1])
axes[1, 1].set_title("Fare box plot")
fig.tight_layout()
fig.savefig(FIG / "univariate_age_fare.png", dpi=120)
plt.show()



Age has **11** IQR outliers (above about 64.8). Fare has **116** IQR outliers (above about 65.6). Fare mean ≈ 32.20, median ≈ 14.45, mode ≈ 8.05, so mean > median > mode and the distribution is **right-skewed**.



## Bivariate analysis

Survival rates by sex, by class, and by sex × class, using boolean masks. Then a 6×6 correlation heatmap on `survived`, `pclass`, `age`, `sibsp`, `parch`, and `fare` only (`adult_male` and `alone` excluded).



In [ ]:
def rate(mask: pd.Series) -> float:
    subset = cleaned.loc[mask, "survived"]
    return round(float(subset.mean()), 4) if len(subset) else None

by_sex = {sex: rate(cleaned["sex"] == sex) for sex in sorted(cleaned["sex"].unique())}
by_pclass = {str(p): rate(cleaned["pclass"] == p) for p in sorted(cleaned["pclass"].unique())}
by_both = {
    f"{sex}_pclass_{p}": rate((cleaned["sex"] == sex) & (cleaned["pclass"] == p))
    for sex in sorted(cleaned["sex"].unique())
    for p in sorted(cleaned["pclass"].unique())
}
print("by_sex", by_sex)
print("by_pclass", by_pclass)
print("by_sex_and_pclass", by_both)

corr = cleaned[STORY_COLUMNS].corr(numeric_only=True).round(4)
pairs = []
cols = list(corr.columns)
for i, left in enumerate(cols):
    for right in cols[i + 1:]:
        value = float(corr.loc[left, right])
        pairs.append({"a": left, "b": right, "correlation": round(value, 4), "abs": abs(value)})
pairs.sort(key=lambda item: item["abs"], reverse=True)
top_corr = pairs[:2]
print("strongest_correlations", top_corr)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation of survived, pclass, age, sibsp, parch, fare")
fig.tight_layout()
fig.savefig(FIG / "correlation_heatmap.png", dpi=120)
plt.show()
corr



**Strongest correlations.** `pclass`–`fare` ≈ −0.55: a higher class number is a cheaper ticket, so these two mostly describe the same wealth split. `sibsp`–`parch` ≈ 0.41: passengers with siblings/spouses also tend to travel with parents/children.



## Multivariate data story

Four charts that argue who was more likely to survive and why.



In [ ]:
sex_rates = cleaned.groupby("sex")["survived"].mean().reset_index()
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=sex_rates, x="sex", y="survived", ax=ax)
ax.set_ylabel("Survival rate")
ax.set_title("Survival rate by sex")
fig.tight_layout()
fig.savefig(FIG / "story_survival_by_sex.png", dpi=120)
plt.show()



Women survived at about 74% and men at about 19%. That gap is larger than the age–survival correlation (−0.07). Sex is the first split in the survival story.



In [ ]:
class_sex = cleaned.groupby(["pclass", "sex"])["survived"].mean().reset_index()
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=class_sex, x="pclass", y="survived", hue="sex", ax=ax)
ax.set_ylabel("Survival rate")
ax.set_title("Survival rate by class and sex")
fig.tight_layout()
fig.savefig(FIG / "story_survival_by_class_sex.png", dpi=120)
plt.show()



First- and second-class women survived above 90%; third-class women about 50%. Men were lower in every class, and a first-class man still survived less often than a third-class woman. Class shifts the level; sex changes the outcome more.



In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=cleaned, x="pclass", y="fare", hue="survived", ax=ax)
ax.set_title("Fare by class and survival")
fig.tight_layout()
fig.savefig(FIG / "story_fare_by_class.png", dpi=120)
plt.show()



Ticket prices fall from first class to third. Survivor boxes sit a little higher inside a class, but the boxes overlap and fare has many IQR outliers. Fare is mostly a noisy restatement of class.



In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=cleaned, x="age", y="fare", hue="survived", alpha=0.7, ax=ax)
ax.set_title("Age vs fare, colored by survival")
fig.tight_layout()
fig.savefig(FIG / "story_age_fare_survival.png", dpi=120)
plt.show()



Survivors appear at every age. The expensive part of the plot (mostly first class) has a denser mix of survivors. There is no single age cutoff that separates outcomes.



## Exploratory standardization check

This z-score check is for EDA only. It does **not** feed into the modeling pipeline, which scales on the training split alone.



In [ ]:
before = cleaned[["age", "fare"]].agg(["mean", "std"]).round(4)
scaled = StandardScaler().fit_transform(cleaned[["age", "fare"]])
scaled_frame = pd.DataFrame(scaled, columns=["age", "fare"])
after = scaled_frame.agg(["mean", "std"]).round(4)
print("before\n", before)
print("after\n", after)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.kdeplot(cleaned["fare"], ax=axes[0], label="original")
sns.kdeplot(scaled_frame["fare"], ax=axes[0], label="z-score")
axes[0].legend()
axes[0].set_title("Fare before and after z-score")
sns.kdeplot(cleaned["age"], ax=axes[1], label="original")
sns.kdeplot(scaled_frame["age"], ax=axes[1], label="z-score")
axes[1].legend()
axes[1].set_title("Age before and after z-score")
fig.tight_layout()
fig.savefig(FIG / "standardization_age_fare.png", dpi=120)
plt.show()



After scaling, age and fare have mean ≈ 0 and standard deviation ≈ 1. The tiny difference from 1 comes from sample vs population standard deviation.



In [ ]:
summary = {
    "profile_shape": list(raw.shape),
    "missing_pct": missing,
    "missing_decisions": decisions,
    "rows_after_cleaning": int(len(cleaned)),
    "age_outliers_iqr": age_outliers,
    "fare_outliers_iqr": fare_outliers,
    "fare_shape": fare_stats,
    "survival": {"by_sex": by_sex, "by_pclass": by_pclass, "by_sex_and_pclass": by_both},
    "correlation": corr.round(4).to_dict(),
    "strongest_correlations": top_corr,
    "standardization": {"before": before.to_dict(), "after": after.to_dict()},
    "class_balance": {
        "survived_rate": round(float(raw["survived"].mean()), 4),
        "counts": {str(k): int(v) for k, v in raw["survived"].value_counts().to_dict().items()},
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Wrote {SUMMARY_PATH.name}")

